# Ch.5 — Monitoring & Observability (Local Stack)

> **Goal:** Instrument a Flask app with Prometheus metrics, deploy the full monitoring stack locally with Docker Compose, and build Grafana dashboards to visualize request rate, latency, and error rate.
>
> **Tech stack:** Python 3.11, Flask, prometheus_client, Docker Compose, Prometheus, Grafana, Alertmanager
>
> **What you'll build:** A production-grade observability stack running entirely on your local machine — no cloud dependencies.

In [ ]:
# Cell 1: Install prometheus_client library

# Install the Prometheus Python client for instrumenting Flask apps
!pip install prometheus-client flask requests

In [ ]:
# Cell 2: Instrument Flask app (counter, histogram, gauge)

# Create a Flask app with Prometheus metrics
# Save this to app.py in your working directory

from flask import Flask, request
from prometheus_client import Counter, Histogram, Gauge, make_wsgi_app
from werkzeug.middleware.dispatcher import DispatcherMiddleware
import time
import random

app = Flask(__name__)

# Define metrics
REQUEST_COUNT = Counter(
    'http_requests_total',
    'Total HTTP requests',
    ['method', 'route', 'status']
)

REQUEST_LATENCY = Histogram(
    'http_request_duration_seconds',
    'HTTP request latency in seconds',
    ['route'],
    buckets=[0.1, 0.5, 1.0, 2.0, 5.0]  # SLA thresholds
)

ACTIVE_CONNECTIONS = Gauge(
    'active_connections',
    'Number of active connections'
)

ERROR_RATE = Gauge(
    'error_rate_percent',
    'Current error rate as percentage'
)

# Middleware to track latency
@app.before_request
def before_request():
    request.start_time = time.time()
    ACTIVE_CONNECTIONS.inc()

@app.after_request
def after_request(response):
    if hasattr(request, 'start_time'):
        latency = time.time() - request.start_time
        REQUEST_LATENCY.labels(route=request.path).observe(latency)
        REQUEST_COUNT.labels(
            method=request.method,
            route=request.path,
            status=response.status_code
        ).inc()
    ACTIVE_CONNECTIONS.dec()
    return response

# Application routes
@app.route('/health')
def health():
    return {'status': 'ok', 'service': 'payment-api'}, 200

@app.route('/api/payment', methods=['POST'])
def payment():
    # Simulate variable processing time
    time.sleep(random.uniform(0.05, 0.3))
    
    # Simulate occasional errors (5% failure rate)
    if random.random() < 0.05:
        return {'error': 'Payment gateway timeout'}, 503
    
    return {'status': 'success', 'transaction_id': random.randint(1000, 9999)}, 200

@app.route('/api/refund', methods=['POST'])
def refund():
    # Refunds are slower (simulate database write)
    time.sleep(random.uniform(0.2, 0.8))
    
    # Simulate occasional errors (10% failure rate)
    if random.random() < 0.10:
        return {'error': 'Refund processing failed'}, 500
    
    return {'status': 'refunded', 'transaction_id': random.randint(1000, 9999)}, 200

# Expose metrics endpoint at /metrics
app.wsgi_app = DispatcherMiddleware(app.wsgi_app, {
    '/metrics': make_wsgi_app()
})

# Save to file
with open('app.py', 'w') as f:
    f.write('''from flask import Flask, request
from prometheus_client import Counter, Histogram, Gauge, make_wsgi_app
from werkzeug.middleware.dispatcher import DispatcherMiddleware
import time
import random

app = Flask(__name__)

REQUEST_COUNT = Counter(
    'http_requests_total',
    'Total HTTP requests',
    ['method', 'route', 'status']
)

REQUEST_LATENCY = Histogram(
    'http_request_duration_seconds',
    'HTTP request latency in seconds',
    ['route'],
    buckets=[0.1, 0.5, 1.0, 2.0, 5.0]
)

ACTIVE_CONNECTIONS = Gauge(
    'active_connections',
    'Number of active connections'
)

@app.before_request
def before_request():
    request.start_time = time.time()
    ACTIVE_CONNECTIONS.inc()

@app.after_request
def after_request(response):
    if hasattr(request, 'start_time'):
        latency = time.time() - request.start_time
        REQUEST_LATENCY.labels(route=request.path).observe(latency)
        REQUEST_COUNT.labels(
            method=request.method,
            route=request.path,
            status=response.status_code
        ).inc()
    ACTIVE_CONNECTIONS.dec()
    return response

@app.route('/health')
def health():
    return {'status': 'ok', 'service': 'payment-api'}, 200

@app.route('/api/payment', methods=['POST'])
def payment():
    time.sleep(random.uniform(0.05, 0.3))
    if random.random() < 0.05:
        return {'error': 'Payment gateway timeout'}, 503
    return {'status': 'success', 'transaction_id': random.randint(1000, 9999)}, 200

@app.route('/api/refund', methods=['POST'])
def refund():
    time.sleep(random.uniform(0.2, 0.8))
    if random.random() < 0.10:
        return {'error': 'Refund processing failed'}, 500
    return {'status': 'refunded', 'transaction_id': random.randint(1000, 9999)}, 200

app.wsgi_app = DispatcherMiddleware(app.wsgi_app, {
    '/metrics': make_wsgi_app()
})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)
''')

print("✅ Flask app saved to app.py")
print("\nMetrics instrumented:")
print("  - http_requests_total (Counter): Tracks request count by method, route, status")
print("  - http_request_duration_seconds (Histogram): Tracks latency distribution by route")
print("  - active_connections (Gauge): Tracks current active connections")
print("\nRoutes:")
print("  - /health (GET): Health check")
print("  - /api/payment (POST): Process payment (5% error rate)")
print("  - /api/refund (POST): Process refund (10% error rate)")
print("  - /metrics (GET): Prometheus metrics endpoint")

In [ ]:
# Cell 3: Deploy Prometheus with docker-compose.yml

# Create Prometheus configuration
prometheus_config = '''
global:
  scrape_interval: 15s  # Scrape targets every 15 seconds
  evaluation_interval: 15s  # Evaluate alerting rules every 15 seconds

scrape_configs:
  - job_name: 'prometheus'
    static_configs:
      - targets: ['localhost:9090']

  - job_name: 'flask-app'
    static_configs:
      - targets: ['flask-app:5000']
'''

with open('prometheus.yml', 'w') as f:
    f.write(prometheus_config)

# Create Docker Compose configuration
docker_compose = '''
version: '3.8'

services:
  flask-app:
    build: .
    container_name: flask-app
    ports:
      - "5000:5000"
    networks:
      - monitoring
    restart: unless-stopped

  prometheus:
    image: prom/prometheus:latest
    container_name: prometheus
    volumes:
      - ./prometheus.yml:/etc/prometheus/prometheus.yml
      - prometheus-data:/prometheus
    command:
      - '--config.file=/etc/prometheus/prometheus.yml'
      - '--storage.tsdb.path=/prometheus'
      - '--storage.tsdb.retention.time=30d'
      - '--web.console.libraries=/usr/share/prometheus/console_libraries'
      - '--web.console.templates=/usr/share/prometheus/consoles'
    ports:
      - "9090:9090"
    networks:
      - monitoring
    restart: unless-stopped

  grafana:
    image: grafana/grafana:latest
    container_name: grafana
    ports:
      - "3000:3000"
    environment:
      - GF_SECURITY_ADMIN_USER=admin
      - GF_SECURITY_ADMIN_PASSWORD=admin
      - GF_USERS_ALLOW_SIGN_UP=false
    volumes:
      - grafana-data:/var/lib/grafana
    networks:
      - monitoring
    restart: unless-stopped

networks:
  monitoring:
    driver: bridge

volumes:
  prometheus-data:
  grafana-data:
'''

with open('docker-compose.yml', 'w') as f:
    f.write(docker_compose)

# Create Dockerfile for Flask app
dockerfile = '''
FROM python:3.11-slim

WORKDIR /app

COPY app.py .

RUN pip install --no-cache-dir flask prometheus-client

EXPOSE 5000

CMD ["python", "app.py"]
'''

with open('Dockerfile', 'w') as f:
    f.write(dockerfile)

print("✅ Configuration files created:")
print("  - prometheus.yml: Prometheus scrape config (scrapes flask-app:5000 every 15s)")
print("  - docker-compose.yml: Full stack (Flask + Prometheus + Grafana)")
print("  - Dockerfile: Flask app container")
print("\n📦 Start the stack with:")
print("  docker compose up -d")
print("\n🌐 Access:")
print("  - Flask app: http://localhost:5000")
print("  - Prometheus: http://localhost:9090")
print("  - Grafana: http://localhost:3000 (user: admin, password: admin)")

In [ ]:
# Cell 4: Verify metrics scraping (Prometheus UI)

# After running 'docker compose up -d', verify Prometheus is scraping metrics

import requests
import time

# Wait for services to start
print("⏳ Waiting for services to start...")
time.sleep(10)

# Check Prometheus targets
try:
    response = requests.get('http://localhost:9090/api/v1/targets')
    data = response.json()
    
    print("\n✅ Prometheus targets:")
    for target in data['data']['activeTargets']:
        job = target['labels']['job']
        state = target['health']
        endpoint = target['scrapeUrl']
        print(f"  - {job}: {state} ({endpoint})")
    
    # Verify Flask /metrics endpoint
    print("\n🔍 Checking Flask /metrics endpoint...")
    metrics_response = requests.get('http://localhost:5000/metrics')
    
    if metrics_response.status_code == 200:
        print("✅ Flask /metrics endpoint is reachable")
        print("\nSample metrics:")
        lines = metrics_response.text.split('\n')
        for line in lines[:20]:  # Show first 20 lines
            if line and not line.startswith('#'):
                print(f"  {line}")
    else:
        print(f"❌ Flask /metrics endpoint returned {metrics_response.status_code}")
    
    print("\n📊 Open Prometheus UI to explore metrics:")
    print("  http://localhost:9090/graph")
    print("\nTry these PromQL queries:")
    print("  - http_requests_total")
    print("  - rate(http_requests_total[5m])")
    print("  - histogram_quantile(0.95, rate(http_request_duration_seconds_bucket[5m]))")
    
except Exception as e:
    print(f"❌ Error connecting to Prometheus: {e}")
    print("\nMake sure the stack is running:")
    print("  docker compose up -d")
    print("\nCheck container status:")
    print("  docker compose ps")

In [ ]:
# Cell 5: Deploy Grafana (already running from docker-compose)

# Grafana is already running from the docker-compose stack
# This cell provides instructions for initial setup

print("📊 Grafana Setup Instructions:")
print("="*60)
print("\n1. Open Grafana: http://localhost:3000")
print("   - Username: admin")
print("   - Password: admin")
print("   (You'll be prompted to change the password on first login)")
print("\n2. Add Prometheus data source:")
print("   - Click 'Add your first data source'")
print("   - Select 'Prometheus'")
print("   - URL: http://prometheus:9090")
print("   - Click 'Save & Test' (should show green checkmark)")
print("\n3. Verify data source connection:")
print("   - Status should show 'Data source is working'")
print("\n✅ Next step: Create a dashboard (Cell 6)")

# Programmatically verify Grafana is reachable
try:
    response = requests.get('http://localhost:3000/api/health')
    if response.status_code == 200:
        print("\n✅ Grafana is running and healthy")
    else:
        print(f"\n⚠️ Grafana returned status {response.status_code}")
except Exception as e:
    print(f"\n❌ Cannot reach Grafana: {e}")

In [ ]:
# Cell 6: Connect Grafana to Prometheus data source

# This cell creates a Grafana data source configuration programmatically
# (Alternative to manual UI setup)

import json

# Grafana data source configuration (JSON)
datasource_config = {
    "name": "Prometheus",
    "type": "prometheus",
    "url": "http://prometheus:9090",
    "access": "proxy",
    "isDefault": True,
    "jsonData": {
        "timeInterval": "15s"
    }
}

# Save to file for manual import
with open('grafana-datasource.json', 'w') as f:
    json.dump(datasource_config, f, indent=2)

print("✅ Grafana data source configuration saved to grafana-datasource.json")
print("\n📝 To add the data source via Grafana API:")
print("\n  curl -X POST \\")
print("    -H 'Content-Type: application/json' \\")
print("    -u admin:admin \\")
print("    -d @grafana-datasource.json \\")
print("    http://localhost:3000/api/datasources")
print("\n✅ Alternatively, add it manually in the Grafana UI:")
print("  1. Configuration (⚙️) → Data Sources → Add data source")
print("  2. Select 'Prometheus'")
print("  3. URL: http://prometheus:9090")
print("  4. Click 'Save & Test'")

In [ ]:
# Cell 7: Create a dashboard (request rate graph)

# Create a Grafana dashboard with request rate, latency, and error rate panels

dashboard_json = {
    "dashboard": {
        "title": "Flask App Monitoring",
        "tags": ["flask", "prometheus"],
        "timezone": "browser",
        "panels": [
            {
                "id": 1,
                "title": "Request Rate (req/sec)",
                "type": "graph",
                "gridPos": {"x": 0, "y": 0, "w": 12, "h": 8},
                "targets": [
                    {
                        "expr": "sum(rate(http_requests_total[5m])) by (route)",
                        "legendFormat": "{{route}}",
                        "refId": "A"
                    }
                ],
                "yaxes": [
                    {"format": "reqps", "label": "Requests/sec"},
                    {"format": "short"}
                ]
            },
            {
                "id": 2,
                "title": "Latency p95 (seconds)",
                "type": "graph",
                "gridPos": {"x": 12, "y": 0, "w": 12, "h": 8},
                "targets": [
                    {
                        "expr": "histogram_quantile(0.95, sum(rate(http_request_duration_seconds_bucket[5m])) by (le, route))",
                        "legendFormat": "p95 {{route}}",
                        "refId": "A"
                    },
                    {
                        "expr": "histogram_quantile(0.50, sum(rate(http_request_duration_seconds_bucket[5m])) by (le, route))",
                        "legendFormat": "p50 {{route}}",
                        "refId": "B"
                    }
                ],
                "yaxes": [
                    {"format": "s", "label": "Latency"},
                    {"format": "short"}
                ]
            },
            {
                "id": 3,
                "title": "Error Rate (%)",
                "type": "graph",
                "gridPos": {"x": 0, "y": 8, "w": 12, "h": 8},
                "targets": [
                    {
                        "expr": "sum(rate(http_requests_total{status=~\"5..\"}[5m])) by (route) / sum(rate(http_requests_total[5m])) by (route) * 100",
                        "legendFormat": "{{route}}",
                        "refId": "A"
                    }
                ],
                "yaxes": [
                    {"format": "percent", "label": "Error Rate"},
                    {"format": "short"}
                ]
            },
            {
                "id": 4,
                "title": "Active Connections",
                "type": "stat",
                "gridPos": {"x": 12, "y": 8, "w": 6, "h": 4},
                "targets": [
                    {
                        "expr": "active_connections",
                        "refId": "A"
                    }
                ]
            },
            {
                "id": 5,
                "title": "Total Requests (1h)",
                "type": "stat",
                "gridPos": {"x": 18, "y": 8, "w": 6, "h": 4},
                "targets": [
                    {
                        "expr": "sum(increase(http_requests_total[1h]))",
                        "refId": "A"
                    }
                ]
            }
        ],
        "refresh": "10s",
        "time": {"from": "now-1h", "to": "now"},
        "timepicker": {"refresh_intervals": ["5s", "10s", "30s", "1m", "5m"]}
    },
    "overwrite": True
}

with open('grafana-dashboard.json', 'w') as f:
    json.dump(dashboard_json, f, indent=2)

print("✅ Grafana dashboard configuration saved to grafana-dashboard.json")
print("\n📊 Dashboard panels:")
print("  1. Request Rate (req/sec) — by route")
print("  2. Latency p95 & p50 (seconds) — by route")
print("  3. Error Rate (%) — 5xx errors by route")
print("  4. Active Connections — current gauge value")
print("  5. Total Requests (1h) — sum over last hour")
print("\n📝 To import the dashboard:")
print("  1. Open Grafana: http://localhost:3000")
print("  2. Click '+' → Import")
print("  3. Upload grafana-dashboard.json")
print("  4. Select 'Prometheus' as data source")
print("  5. Click 'Import'")
print("\n✅ The dashboard will auto-refresh every 10 seconds")

In [ ]:
# Cell 8: Add alerting rules (Prometheus Alertmanager)

# Create Prometheus alerting rules
alerting_rules = '''
groups:
  - name: flask_app_alerts
    interval: 30s
    rules:
      - alert: HighErrorRate
        expr: sum(rate(http_requests_total{status=~"5.."}[5m])) by (route) / sum(rate(http_requests_total[5m])) by (route) > 0.05
        for: 2m
        labels:
          severity: warning
        annotations:
          summary: "High error rate on {{ $labels.route }}"
          description: "Error rate is {{ $value | humanizePercentage }} for route {{ $labels.route }}"
      
      - alert: HighLatency
        expr: histogram_quantile(0.95, sum(rate(http_request_duration_seconds_bucket[5m])) by (le, route)) > 1.0
        for: 5m
        labels:
          severity: warning
        annotations:
          summary: "High latency on {{ $labels.route }}"
          description: "p95 latency is {{ $value }}s for route {{ $labels.route }}"
      
      - alert: ServiceDown
        expr: up{job="flask-app"} == 0
        for: 1m
        labels:
          severity: critical
        annotations:
          summary: "Flask app is down"
          description: "Flask app has been down for more than 1 minute"
'''

with open('alerting_rules.yml', 'w') as f:
    f.write(alerting_rules)

# Update prometheus.yml to include alerting rules
prometheus_config_with_alerts = '''
global:
  scrape_interval: 15s
  evaluation_interval: 15s

rule_files:
  - "alerting_rules.yml"

scrape_configs:
  - job_name: 'prometheus'
    static_configs:
      - targets: ['localhost:9090']

  - job_name: 'flask-app'
    static_configs:
      - targets: ['flask-app:5000']
'''

with open('prometheus.yml', 'w') as f:
    f.write(prometheus_config_with_alerts)

print("✅ Alerting rules saved to alerting_rules.yml")
print("✅ prometheus.yml updated to include alerting rules")
print("\n🚨 Configured alerts:")
print("  1. HighErrorRate: Fires when error rate > 5% for 2 minutes")
print("  2. HighLatency: Fires when p95 latency > 1.0s for 5 minutes")
print("  3. ServiceDown: Fires when Flask app is unreachable for 1 minute")
print("\n📝 To apply the changes:")
print("  1. Update docker-compose.yml to mount alerting_rules.yml")
print("  2. Restart Prometheus:")
print("     docker compose restart prometheus")
print("\n🔍 View alerts in Prometheus UI:")
print("  http://localhost:9090/alerts")
print("\n⚠️ To route alerts to Slack/email, add Alertmanager to docker-compose.yml")

In [ ]:
# Cell 9: Simulate traffic and observe metrics

# Generate synthetic traffic to populate the dashboard

import concurrent.futures
import random

def send_request(url, method='GET', json_data=None):
    """Send a single request and return status code"""
    try:
        if method == 'POST':
            response = requests.post(url, json=json_data, timeout=5)
        else:
            response = requests.get(url, timeout=5)
        return response.status_code
    except Exception as e:
        return f"Error: {e}"

def generate_traffic(duration_seconds=60, requests_per_second=5):
    """Generate synthetic traffic for a specified duration"""
    urls = [
        ('http://localhost:5000/health', 'GET', None),
        ('http://localhost:5000/api/payment', 'POST', {'amount': 100}),
        ('http://localhost:5000/api/refund', 'POST', {'transaction_id': 1234}),
    ]
    
    total_requests = duration_seconds * requests_per_second
    status_counts = {}
    
    print(f"🚀 Generating {requests_per_second} req/sec for {duration_seconds} seconds...")
    print(f"📊 Total requests: {total_requests}")
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
        futures = []
        
        for i in range(total_requests):
            # Randomly select a route (weighted: 60% health, 30% payment, 10% refund)
            route_weights = [0.6, 0.3, 0.1]
            url, method, json_data = random.choices(urls, weights=route_weights)[0]
            
            # Submit request
            future = executor.submit(send_request, url, method, json_data)
            futures.append(future)
            
            # Rate limiting: sleep to maintain requests_per_second
            if (i + 1) % requests_per_second == 0:
                time.sleep(1)
            
            # Progress update every 50 requests
            if (i + 1) % 50 == 0:
                print(f"  Progress: {i + 1}/{total_requests} requests sent")
        
        # Collect results
        for future in concurrent.futures.as_completed(futures):
            status = future.result()
            status_counts[status] = status_counts.get(status, 0) + 1
    
    print("\n✅ Traffic generation complete!")
    print("\n📊 Response status distribution:")
    for status, count in sorted(status_counts.items()):
        percentage = (count / total_requests) * 100
        print(f"  {status}: {count} ({percentage:.1f}%)")
    
    print("\n🔍 View the results in Grafana:")
    print("  http://localhost:3000")
    print("\n💡 Metrics you should see:")
    print("  - Request rate spike in the last minute")
    print("  - Latency histogram with realistic distribution")
    print("  - ~5% error rate on /api/payment")
    print("  - ~10% error rate on /api/refund")

# Generate traffic
generate_traffic(duration_seconds=60, requests_per_second=5)

In [ ]:
# Cell 10: Export dashboard as JSON

# The dashboard JSON was already created in Cell 7
# This cell provides instructions for exporting from Grafana UI

print("📤 Export Dashboard from Grafana UI:")
print("="*60)
print("\n1. Open your dashboard in Grafana: http://localhost:3000")
print("\n2. Click the 'Share' icon (next to dashboard title)")
print("\n3. Click 'Export' tab")
print("\n4. Click 'Save to file' → downloads dashboard JSON")
print("\n5. Commit the JSON to version control:")
print("   git add grafana-dashboard.json")
print("   git commit -m 'Add monitoring dashboard'")
print("\n✅ Benefits of version-controlled dashboards:")
print("  - Track changes over time (git history)")
print("  - Review dashboard changes in pull requests")
print("  - Rollback to previous dashboard versions")
print("  - Share dashboards across teams")
print("  - Automate dashboard deployment (Infrastructure as Code)")
print("\n📝 Alternative: Use Grafana API to export programmatically:")
print("\n  # Get dashboard UID from Grafana UI (Dashboard Settings → JSON Model)")
print("  curl -X GET \\")
print("    -H 'Authorization: Bearer <api-key>' \\")
print("    http://localhost:3000/api/dashboards/uid/<dashboard-uid>")
print("\n🎯 Summary — What You Built:")
print("="*60)
print("✅ Flask app instrumented with Prometheus metrics")
print("✅ Prometheus scraping metrics every 15 seconds")
print("✅ Grafana dashboard with request rate, latency, and error rate")
print("✅ Alerting rules for high error rate, high latency, and service down")
print("✅ Full monitoring stack running locally with Docker Compose")
print("\n🚀 Next Steps:")
print("  - Add distributed tracing with OpenTelemetry (Cell 11 in supplement)")
print("  - Deploy to production with Infrastructure as Code (Ch.6)")
print("  - Integrate Alertmanager with Slack/PagerDuty")
print("  - Add custom business metrics (conversion rate, revenue/hour)")
print("\n🧹 To stop the stack:")
print("  docker compose down")
print("\n🗑️ To remove all data (Prometheus TSDB, Grafana dashboards):")
print("  docker compose down -v")